# 01 — Fondamentaux Azure & PaaS

**Objectif :** comprendre les concepts du cloud (façon AZ-900), les bases de l'IA, et **déployer concrètement** deux services PaaS Azure depuis ce notebook avec l'`az cli` :

1. **App Service** — héberger une application web sans gérer de serveur.
2. **Microsoft Foundry** — la plateforme Azure pour construire des applications et des agents IA.

À la fin du notebook, vous aurez :
- compris ce qu'est un Resource Group, une région, un SKU, le modèle PaaS ;
- compris les grandes familles de l'IA (ML, DL, GenAI) et le vocabulaire de la GenAI (tokens, LLM/SLM, RAG, agents) ;
- créé toutes ces ressources via la ligne de commande ;
- supprimé proprement tout ce que vous avez créé.

> 💡 Exécutez **chaque cellule l'une après l'autre** et lisez les explications. Tout ce qui commence par `!` dans une cellule Python est exécuté dans le shell.
>
> 📈 Le **monitoring** de l'application (Application Insights) est traité dans le **notebook 02**, qui se branche sur la Web App créée ici.

## 1. Qu'est-ce que le cloud ?

Le **cloud computing**, c'est consommer des ressources informatiques (calcul, stockage, réseau, IA…) **à la demande, via Internet**, facturées à l'usage. Plus besoin d'acheter des serveurs physiques.

### Les trois grands modèles de service

| Modèle | Ce que **vous** gérez | Ce que **le cloud** gère | Exemple Azure |
|--------|----------------------|--------------------------|---------------|
| **IaaS** (Infrastructure as a Service) | OS, runtimes, code, données | Hyperviseur, matériel, réseau | Azure **Virtual Machines** |
| **PaaS** (Platform as a Service) | Code, données, configuration | OS, runtime, patchs, scaling | Azure **App Service**, **Functions** |
| **SaaS** (Software as a Service) | Données et utilisateurs | Tout le reste | **Microsoft 365**, **Dynamics** |

👉 Dans ce notebook, on se concentre sur le **PaaS** : on déploie du code sans se soucier des VMs.

### Les modèles de déploiement

- **Public cloud** — partagé, hébergé par Microsoft (le cas par défaut, et celui qu'on utilise).
- **Privé** — chez vous ou un hébergeur dédié.
- **Hybride** — un mix des deux.

### Pourquoi le cloud ?

- **Élasticité** : on monte/descend la capacité en quelques secondes.
- **OpEx vs CapEx** : on paie à l'usage, pas en investissement.
- **Couverture mondiale** : déployable dans 60+ régions.
- **Sécurité & conformité** intégrées (Entra ID, RBAC, certifications).

## 2. Concepts Azure à connaître

Avant de cliquer, comprenez la **hiérarchie Azure** :

```
Tenant Microsoft Entra ID
 └── Management Group
      └── Subscription (abonnement, l'unité de facturation)
           └── Resource Group (dossier logique)
                └── Resources (App Service, BDD, etc.)
```

- **Subscription** : où sont facturées les ressources.
- **Resource Group (RG)** : un conteneur logique. **Toutes** vos ressources doivent appartenir à un RG. Supprimer un RG supprime **tout** ce qu'il contient — pratique pour le nettoyage.
- **Région** (ex. `francecentral`, `westeurope`, `eastus`) : le datacenter physique. Choisissez une région proche de vos utilisateurs.
- **SKU** (Stock Keeping Unit) : la « taille »/tier d'un service (ex. `F1` gratuit, `B1` basic, `S1` standard…).
- **Azure Resource Manager (ARM)** : l'API unifiée qui orchestre tout. L'`az cli`, le portail, Bicep, Terraform passent tous par ARM.

### Quelques services PaaS phares

| Service | À quoi ça sert |
|---------|---------------|
| **App Service** | Héberger des applis web (Node, .NET, Python, Java…) sans gérer de VM. |
| **Azure Functions** | Exécuter du code à l'événement (serverless). |
| **Application Insights** | Monitorer perfs, erreurs, logs d'une appli. |
| **Azure SQL / Cosmos DB** | Bases de données managées. |
| **Key Vault** | Stocker secrets, clés, certificats. |
| **Microsoft Foundry** | Construire, déployer et évaluer des agents et applications IA. |

## 3. Concepts d'IA & Microsoft Foundry

Avant d'attaquer le « comment » (déployer une ressource Foundry avec l'`az cli`), on prend un moment pour comprendre **ce qu'est l'IA aujourd'hui**, les grandes familles de modèles, et **où Microsoft Foundry se positionne** dans cet écosystème.

Ce chapitre est volontairement théorique : pas de code, juste les bases conceptuelles qui rendront la suite beaucoup plus claire.

---

### 3.1 De la machine learning à l'IA générative

#### 3.1.1 Vue d'ensemble — IA ⊃ ML ⊃ DL ⊃ GenAI

Ces termes sont **emboîtés** comme des poupées russes :

```
┌────────────────────────────────────────────────────────────┐
│ Intelligence Artificielle (IA)                             │
│  toute approche qui imite des capacités humaines           │
│  (logique, perception, raisonnement, langage…)             │
│                                                            │
│  ┌──────────────────────────────────────────────────────┐  │
│  │ Machine Learning (ML)                                │  │
│  │  apprendre à partir de données, sans être           │  │
│  │  programmé règle par règle                           │  │
│  │                                                      │  │
│  │  ┌────────────────────────────────────────────────┐  │  │
│  │  │ Deep Learning (DL)                             │  │  │
│  │  │  ML avec des réseaux de neurones profonds      │  │  │
│  │  │                                                │  │  │
│  │  │  ┌──────────────────────────────────────────┐  │  │  │
│  │  │  │ IA générative (GenAI)                    │  │  │  │
│  │  │  │  DL qui *génère* (texte, image, code…)   │  │  │  │
│  │  │  │  → LLM, diffusion models, etc.           │  │  │  │
│  │  │  └──────────────────────────────────────────┘  │  │  │
│  │  └────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────────┘
```

Retenez : **toute GenAI est du Deep Learning, mais tout le ML n'est pas du Deep Learning**. Et on a fait de l'IA bien avant le ML (systèmes experts, règles, recherche de chemin…).

---

#### 3.1.2 Machine Learning — les fondations

Un modèle de ML apprend une fonction `f(X) → y` à partir d'exemples. On distingue **trois grandes familles** d'apprentissage :

| Type | Données | Exemple concret | Algos typiques |
|------|---------|-----------------|----------------|
| **Supervisé** | Données + **labels** (la réponse attendue) | Prédire le prix d'un appart à partir de sa surface | Régression linéaire, régression logistique, arbres de décision, Random Forest, Gradient Boosting (XGBoost), SVM |
| **Non supervisé** | Données seules, pas de labels | Segmenter une base clients en groupes similaires | K-means, DBSCAN, ACP/PCA, autoencodeurs |
| **Par renforcement** | Un agent qui interagit avec un environnement et reçoit des récompenses | Apprendre à jouer aux échecs, piloter un robot | Q-learning, PPO, RLHF (utilisé pour affiner ChatGPT) |

##### Les deux grandes tâches en supervisé

- **Régression** → prédire une valeur **continue** (ex. prix, température, durée).  
  Exemple : *« combien va coûter cet appartement ? »* → 285 000 €.  
  Algo le plus simple : la **régression linéaire** `y = a·x + b`.

- **Classification** → prédire une **catégorie**.  
  Exemple : *« cet email est-il un spam ? »* → oui / non.  
  Algos : régression **logistique** (malgré son nom, c'est de la classif), arbres, SVM…

##### Le cycle d'un projet ML

1. **Collecter** des données.
2. **Préparer** : nettoyer, normaliser, *feature engineering*.
3. **Entraîner** : on découpe en jeux *train / validation / test*. Le modèle ajuste ses **paramètres** pour minimiser une *fonction de perte* (loss).
4. **Évaluer** : précision, rappel, F1, RMSE… selon la tâche.
5. **Déployer** et **monitorer** la dérive (le monde change, le modèle vieillit).

> 🧠 Vocabulaire-clé : *features* (entrées), *labels* (sorties attendues), *overfitting* (le modèle apprend par cœur le train et rate le test), *hyperparamètres* (réglages choisis avant l'entraînement, ex. nombre d'arbres).

---

#### 3.1.3 Deep Learning & réseaux de neurones

Un **neurone artificiel** fait une opération très simple : il prend des entrées, les pondère, additionne, et passe le résultat dans une fonction non-linéaire (*activation*, ex. `ReLU`).

```
x1 ─┐
x2 ─┼─► [Σ wi·xi + b] ─► activation ─► sortie
x3 ─┘
```

Empilez des milliers de ces neurones en **couches** → vous obtenez un **réseau de neurones profond** (deep = beaucoup de couches). L'entraînement utilise la **rétropropagation du gradient** (*backprop*) : on calcule à quel point chaque poids a contribué à l'erreur, et on l'ajuste un peu dans le bon sens. Répétez des millions de fois.

##### Pourquoi le DL a explosé vers 2012 ?

Trois facteurs simultanés :
1. **Données massives** (Internet, ImageNet, etc.).
2. **GPU** capables de paralléliser les calculs matriciels.
3. **Algorithmes** matures (ReLU, dropout, batch normalization…).

##### Architectures classiques à connaître

| Archi | Bonne pour | Exemple |
|------|------------|---------|
| **CNN** (Convolutional Neural Network) | Images | Détection d'objets, vision médicale |
| **RNN / LSTM / GRU** | Séquences (texte, séries temporelles) | Traduction d'avant 2017, prévision météo |
| **Transformer** ⭐ | Séquences, mais en **parallèle** grâce à l'**attention** | GPT, BERT, Llama, Mistral, Phi… |

Le **Transformer** (article *« Attention is All You Need »*, Google 2017) est l'innovation qui a rendu possible l'IA générative moderne. Son mécanisme d'**attention** permet à chaque mot d'« regarder » tous les autres mots de la séquence pour comprendre le contexte — et tout ça en parallèle, ce qui scale très bien sur GPU.

---

#### 3.1.4 L'arrivée de l'IA générative

L'**IA générative** est un Deep Learning dont la sortie n'est plus une classe ou un nombre, mais **du contenu** (texte, code, image, audio, vidéo, 3D…).

##### Familles de modèles génératifs

| Sortie | Famille | Exemples |
|--------|---------|----------|
| Texte / code | **LLM** (Large Language Model), basés sur Transformer | GPT-5, Claude, Llama 3, Mistral, Phi |
| Image | **Diffusion models** | DALL·E, Stable Diffusion, FLUX |
| Audio / parole | TTS, Whisper-like | Azure Speech, ElevenLabs |
| Vidéo | Diffusion vidéo | Sora |
| Multimodal | Modèles « omni » | GPT-4o, Gemini |

##### Comment un LLM fonctionne (en 1 minute)

1. On lui donne un **prompt** (du texte).
2. Il prédit **le prochain token** le plus probable.
3. Il rajoute ce token au prompt, et recommence.
4. Il s'arrête quand il génère un token spécial *fin* ou atteint la limite.

C'est tout. Un LLM est, fondamentalement, **une machine très sophistiquée à prédire le token suivant**.

---

#### 3.1.5 Concepts essentiels de la GenAI

##### Token

Un **token** ≈ un morceau de mot. En anglais, **~4 caractères** ou **~0,75 mot** par token. En français, c'est un peu plus coûteux.

| Texte | ~ Tokens |
|-------|---------:|
| `Hello world` | 2 |
| `Bonjour le monde` | 4 |
| `import pandas as pd` | 5 |

> 💰 **La facturation des LLM se fait au token** (input + output). Et la **context window** (mémoire courante du modèle) se mesure aussi en tokens. Compter ses tokens, c'est compter ses sous.

##### Context window

C'est la **quantité de tokens** que le modèle peut « voir » en une fois (prompt + historique + sortie).
- GPT-4o : 128k tokens
- GPT-4.1 / GPT-5 : jusqu'à 1M+
- Claude Sonnet : 200k+
- Llama 3 : 8k → 128k selon variante

Plus la fenêtre est grande, plus on peut y caser de contexte (documents, historique de conversation) — mais aussi plus c'est cher et lent.

##### LLM vs SLM

| | **LLM** (Large) | **SLM** (Small) |
|---|---|---|
| Taille (paramètres) | 70B → 1T+ | < 10B (souvent 1–7B) |
| Hébergement | Cloud, GPU coûteux | Cloud léger, edge, voire **on-device** (téléphone, navigateur) |
| Latence | Élevée | Faible |
| Coût | Élevé | Très bas |
| Cas d'usage | Tâches complexes, raisonnement, créatif | Tâches ciblées, extraction, classification, on-device |
| Exemples | GPT-5, Claude Opus, Llama 70B | **Phi-4**, Llama 3.2 1B/3B, Mistral 7B, Gemma 2 |

> 🎯 La tendance 2025-2026 : combiner **un SLM rapide en première ligne** + un **LLM puissant en escalade** quand c'est nécessaire. C'est plus rapide et bien moins cher.

##### Embeddings

Un **embedding**, c'est un **vecteur** (liste de nombres, ex. 1536 dimensions) qui représente le **sens** d'un texte. Deux textes proches sémantiquement ont des vecteurs proches (distance cosine faible).

Usage typique : la **recherche sémantique** et le **RAG** (cf. ci-dessous).

##### Training vs Fine-tuning vs Inference

- **Pre-training** : entraîner un modèle de zéro sur des téraoctets de texte. Coûte des **millions de dollars**. Quelques acteurs au monde.
- **Fine-tuning** : ré-entraîner légèrement un modèle pré-entraîné sur vos données pour le spécialiser. Beaucoup moins cher.
- **Inference** : *utiliser* le modèle (lui poser une question). C'est ce qu'on fait 99 % du temps.

##### Prompt, system prompt, RAG, agents — la pyramide

1. **Prompt** : ce qu'on écrit au modèle.
2. **System prompt** : instructions de cadrage (« tu es un assistant comptable… »).
3. **Few-shot** : on glisse 2-3 exemples dans le prompt pour guider la réponse.
4. **RAG** (Retrieval-Augmented Generation) : avant de répondre, on **cherche** dans une base de documents les passages pertinents, et on les injecte dans le prompt. → Le modèle répond avec des **sources** et sans hallucinations.
5. **Agent** : un LLM + des **outils** (fonctions à appeler) + une **boucle** de raisonnement. L'agent décide quoi faire, appelle des outils, lit les résultats, et continue jusqu'à atteindre l'objectif.

##### Hallucinations

Un LLM peut **inventer** des faits avec aplomb. C'est inhérent au mécanisme. Les parades : RAG, *grounding*, citation des sources, évaluation systématique, et un humain dans la boucle pour les sujets sensibles.

---

> ✅ **À ce stade vous savez :** ce qu'est l'IA, le ML, le DL, la GenAI ; les algos de régression / classification ; comment marche un réseau de neurones ; ce qu'est un LLM, un SLM, un token, une context window, un embedding ; et le vocabulaire prompt / RAG / agent.
>
> On peut passer à **comment faire tout ça concrètement sur Azure** → section 3.2.

### 3.2 Microsoft Foundry — la plateforme IA d'Azure

Maintenant qu'on a les concepts, **où fait-on tout ça concrètement chez Microsoft ?** Réponse : **Microsoft Foundry** (anciennement *Azure AI Foundry / Azure AI Studio*).

Foundry, c'est **la plateforme unifiée** pour bâtir et opérer des applications IA — du prototype à la production.

#### Ce qu'on y trouve

| Brique Foundry | À quoi ça correspond dans le 3.1 |
|----------------|----------------------------------|
| **Model catalog** | Le supermarché des modèles : GPT-5, Llama, Mistral, **Phi (SLM Microsoft)**, DeepSeek, modèles d'images, de speech… |
| **Model deployments** | L'**inference** d'un modèle exposé en endpoint HTTPS (pay-per-token). |
| **Playgrounds** | Tester un prompt / une conversation sans écrire une ligne de code. |
| **Agents** | Construire des **agents** (LLM + outils + boucle) en quelques clics ou en SDK. |
| **Indexes** | Bases vectorielles d'**embeddings** → support du **RAG**. |
| **Datasets** | Jeux de données pour évaluer ou *fine-tuner*. |
| **Evaluations** | Mesurer la qualité d'un modèle ou d'un agent (groundedness, relevance, safety…). |
| **Tracing** | Voir ce que fait l'agent étape par étape — exporté vers **Application Insights** (qu'on branchera dans le notebook 02). |
| **Content safety** | Filtres pour bloquer prompts malveillants et sorties à risque. |
| **Fine-tuning** | Spécialiser un modèle sur vos données. |

#### Hiérarchie des ressources

```
Foundry resource  (Azure resource, kind = AIServices, projet management activé)
 │
 ├── Foundry project #1   (« default »)
 │    ├── model deployments (gpt-5-mini, text-embedding-3-large, …)
 │    ├── agents
 │    ├── indexes / datasets
 │    └── evaluations / traces
 │
 └── Foundry project #2   (équipe / produit différent)
      └── …
```

- **Une seule ressource Foundry** peut héberger **plusieurs projets** — pratique pour isoler des équipes tout en mutualisant les modèles déployés et la sécurité.
- Les **projets** sont des sous-ressources Azure, ils ont leur propre RBAC et leur propre identité.
- L'authentification se fait via **Microsoft Entra ID** (`DefaultAzureCredential` côté SDK). Pas de clé API à se trimballer.

#### Pourquoi Foundry plutôt que d'appeler l'API OpenAI en direct ?

- **Sécurité & compliance Azure** : tenant Entra ID, RBAC, private endpoints, content safety, journaux d'audit.
- **Catalogue multi-modèles** : on n'est pas verrouillé sur un fournisseur. Llama, Mistral, Phi, GPT… au même endroit, même API d'inférence.
- **Intégration native** avec App Service, Application Insights, Key Vault, Cosmos DB, AI Search…
- **Tooling fini** : évaluations, traces, content safety, fine-tuning — au lieu de tout réécrire à la main.

#### Ce qu'on va faire dans la suite du notebook

Plus loin (chapitre 8), on va **provisionner avec l'`az cli`** :
1. Une **ressource Foundry** (`kind = AIServices`, project management activé).
2. Un **custom subdomain** (requis pour l'authentification Entra ID).
3. Un **projet** dans cette ressource.

Puis on ouvrira https://ai.azure.com pour visualiser tout ça. Le **monitoring de la Web App** sera couvert dans le notebook 02, et le déploiement des modèles + agents dans les suivants.

📚 Pour aller plus loin : https://learn.microsoft.com/azure/foundry/

## 4. Pré-requis : Azure CLI & SDK Python

On vérifie que l'`az cli` est installée (version recommandée **≥ 2.86**). Si la commande ci-dessous échoue ou affiche une version plus ancienne, mettez-la à jour : https://learn.microsoft.com/cli/azure/install-azure-cli

- Mettre à jour l'`az cli` en place : `az upgrade`
- Lister les extensions installées : `az extension list -o table`

In [ ]:
!az --version

### SDK Python Azure (dernières versions stables)

On installe les SDK Python utilisés dans ce notebook et les suivants. Les versions sont **épinglées** aux dernières stables vérifiées sur PyPI (mai 2026). Un `requirements.txt` est aussi présent à la racine du repo si vous préférez `pip install -r ../requirements.txt`.

| Package | Rôle | Version min |
|---------|------|------------:|
| `azure-identity` | Authentification (DefaultAzureCredential, MSI…) | `1.25.3` |
| `azure-mgmt-cognitiveservices` | Gestion (CRUD) des ressources Foundry / Cognitive Services | `14.1.0` |
| `azure-ai-projects` | SDK plan **data** pour les projets Foundry | `2.1.0` |
| `azure-ai-agents` | SDK pour créer/invoquer des agents Foundry | `1.1.0` |
| `azure-monitor-opentelemetry` | Tracing + logs vers Application Insights via OpenTelemetry | `1.8.8` |

In [ ]:
%pip install --upgrade \
    "azure-identity>=1.25.3" \
    "azure-mgmt-cognitiveservices>=14.1.0" \
    "azure-ai-projects>=2.1.0" \
    "azure-ai-agents>=1.1.0" \
    "azure-monitor-opentelemetry>=1.8.8"

### Connexion à Azure

La commande suivante ouvre une fenêtre de navigateur pour vous authentifier. Si vous êtes sur une machine sans navigateur, utilisez `az login --use-device-code`.

In [ ]:
!az login

### Lister et choisir votre abonnement

Si vous avez plusieurs abonnements, sélectionnez celui sur lequel vous voulez travailler.

In [ ]:
!az account list --output table

In [ ]:
# 👉 Remplacez par le nom ou l'ID de votre abonnement, puis exécutez.
SUBSCRIPTION = "<votre-subscription-name-ou-id>"
!az account set --subscription "{SUBSCRIPTION}"
!az account show --output table

## 5. Définir nos variables

On centralise ici les noms et la région. Les ressources Azure ont des règles de nommage : minuscules, sans accents, et le suffixe `SUFFIX` rend les noms uniques globalement (App Service et Foundry exposent un DNS public).

In [ ]:
import os, random, string

# Suffixe aléatoire pour rendre les noms uniques. Régénérez si conflit.
SUFFIX = "".join(random.choices(string.ascii_lowercase + string.digits, k=5))

LOCATION   = "francecentral"           # région Azure
RG         = f"rg-stagiaire-{SUFFIX}"  # resource group
PLAN       = f"asp-stagiaire-{SUFFIX}" # app service plan
WEBAPP     = f"web-stagiaire-{SUFFIX}" # web app (doit être unique mondialement)
FOUNDRY    = f"aif-stagiaire-{SUFFIX}" # ressource Foundry (Cognitive Services AIServices)
PROJECT    = "my-foundry-project"      # projet Foundry

# Exporte vers l'environnement pour les cellules shell (!az ...)
for k, v in {
    "LOCATION": LOCATION, "RG": RG, "PLAN": PLAN, "WEBAPP": WEBAPP,
    "FOUNDRY": FOUNDRY, "PROJECT": PROJECT,
}.items():
    os.environ[k] = v

print(f"Suffixe unique : {SUFFIX}")
print(f"Resource Group : {RG}")
print(f"Web App        : https://{WEBAPP}.azurewebsites.net")
print(f"Foundry        : {FOUNDRY}")

# 💡 Conservez ce suffixe si vous prévoyez d'enchaîner sur le notebook 02 (monitoring) :
#    il vous permettra de retrouver les mêmes ressources.
print(f"\n👉 Pour le notebook suivant, notez ce suffixe : {SUFFIX}")

## 6. Créer le Resource Group

C'est notre « dossier » Azure. Toutes les ressources ci-dessous y seront créées. À la fin, on supprimera ce RG et tout sera nettoyé en une commande.

In [ ]:
!az group create --name $RG --location $LOCATION --output table

## 7. App Service — héberger une application web

**App Service** est le PaaS d'hébergement web d'Azure. On fournit du code (ou un conteneur), Azure s'occupe de l'OS, du runtime, des patchs, du HTTPS, du scaling.

Deux objets à créer :

1. **App Service Plan** — la « machine » sous-jacente (CPU/RAM). On prend `F1` (gratuit, Linux).
2. **Web App** — l'application qui tourne sur ce plan.

> Le plan `F1` a des limites : 1 Go RAM, 60 min de CPU/jour, pas d'auto-scale. Parfait pour apprendre.

In [ ]:
# 7.1 Créer l'App Service Plan (Linux, SKU gratuit F1)
!az appservice plan create \
    --name $PLAN \
    --resource-group $RG \
    --sku F1 \
    --is-linux \
    --output table

In [ ]:
# 7.2 Créer la Web App avec un runtime Python 3.13 (dernière stable largement supportée par l'écosystème)
#     Pour voir la liste complète : !az webapp list-runtimes --os linux -o tsv | findstr PYTHON
!az webapp create \
    --name $WEBAPP \
    --resource-group $RG \
    --plan $PLAN \
    --runtime "PYTHON:3.13" \
    --output table

In [ ]:
# 7.3 Récupérer l'URL publique de l'app
!az webapp show --name $WEBAPP --resource-group $RG --query defaultHostName -o tsv

Ouvrez l'URL dans votre navigateur. Vous devriez voir la page d'accueil par défaut « Your web app is running and waiting for your content ». Pour déployer du code par la suite, on utiliserait `az webapp up`, un déploiement Git, ou un pipeline Azure DevOps / GitHub Actions.

## 8. Provisionner Microsoft Foundry

On a vu en **3.2** ce qu'est Foundry et son modèle de ressources. On passe maintenant à la pratique : créer la ressource, son sous-domaine personnalisé, et un projet, avec l'`az cli`.

Trois étapes :

1. **Créer la ressource Foundry** (`kind = AIServices`, project management activé).
2. **Configurer un custom subdomain** — requis pour l'authentification Entra ID.
3. **Créer un projet** dans cette ressource.

📚 Doc : https://learn.microsoft.com/azure/foundry/how-to/create-projects

In [ ]:
# 8.1 Créer la ressource Foundry (Cognitive Services kind=AIServices)
#     --allow-project-management active la possibilité de créer des projets dessus.
!az cognitiveservices account create \
    --name $FOUNDRY \
    --resource-group $RG \
    --kind AIServices \
    --sku S0 \
    --location $LOCATION \
    --allow-project-management true \
    --yes \
    --output table

In [ ]:
# 8.2 Attribuer un custom sub-domain (requis pour les appels Foundry, doit être unique mondialement)
!az cognitiveservices account update \
    --name $FOUNDRY \
    --resource-group $RG \
    --custom-domain $FOUNDRY \
    --output table

In [ ]:
# 8.3 Créer un projet Foundry dans la ressource
!az cognitiveservices account project create \
    --name $FOUNDRY \
    --resource-group $RG \
    --project-name $PROJECT \
    --location $LOCATION \
    --output table

In [ ]:
# 8.4 Vérifier le projet
!az cognitiveservices account project show \
    --name $FOUNDRY \
    --resource-group $RG \
    --project-name $PROJECT \
    --output jsonc

🎉 Ouvrez ensuite **https://ai.azure.com** : vous y retrouverez votre projet `my-foundry-project`. Dans les prochains notebooks, on y déploiera un modèle et on y créera un premier agent.

## 9. Vue d'ensemble de ce qu'on a créé

Listons toutes les ressources du Resource Group pour visualiser ce qui a été déployé.

In [ ]:
!az resource list --resource-group $RG --output table

## 10. 🧹 Nettoyage — IMPORTANT

Pour éviter toute facturation inutile, supprimez le Resource Group quand vous avez fini. Une seule commande supprime **toutes** les ressources créées dans ce notebook.

> ⚠️ L'opération est **irréversible**. Confirmez bien que `RG` désigne le bon groupe avant d'exécuter.
>
> 💡 Si vous comptez enchaîner sur le notebook **02 (Application Insights)**, **gardez** le Resource Group : ce notebook ajoutera le monitoring **par-dessus** la même Web App.

In [ ]:
print(f"Vous allez supprimer le resource group : {RG}")
# Décommentez la ligne ci-dessous pour exécuter la suppression :
# !az group delete --name $RG --yes --no-wait

## Récap & prochaines étapes

Vous savez maintenant :
- ce qu'est le **cloud**, IaaS / PaaS / SaaS, et la hiérarchie Azure ;
- les bases de l'**IA** : ML, DL, GenAI, tokens, LLM/SLM, embeddings, RAG, agents ;
- ce qu'est **Microsoft Foundry** et comment il s'inscrit dans cet écosystème ;
- vous connecter avec `az login` et choisir un abonnement ;
- créer un **Resource Group** ;
- déployer une **Web App** sur **App Service** ;
- provisionner une ressource et un projet **Microsoft Foundry** ;
- nettoyer proprement.

**Prochains notebooks :**
1. **02 — Monitoring de l'application avec Application Insights** : ajouter la télémétrie sur la Web App créée ici.
2. Identité Azure : Microsoft Entra ID, RBAC, Managed Identity, Key Vault.
3. Déployer un modèle (GPT, Phi…) et un premier agent dans le projet Foundry.
4. Infrastructure as Code avec Bicep — refaire tout ce notebook en déclaratif.
5. Tracing & évaluation d'agents Foundry.

📚 Pour aller plus loin :
- Parcours AZ-900 : https://learn.microsoft.com/training/paths/microsoft-azure-fundamentals-describe-cloud-concepts/
- Parcours AI-900 (concepts IA) : https://learn.microsoft.com/training/paths/get-started-with-artificial-intelligence-on-azure/
- Doc App Service : https://learn.microsoft.com/azure/app-service/
- Doc Microsoft Foundry : https://learn.microsoft.com/azure/foundry/
- *Attention is All You Need* (l'article fondateur du Transformer) : https://arxiv.org/abs/1706.03762